# 多输入输出通道

## 实现一下多输入通道互相关运算

In [1]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X, K):          # 这里输入数据的格式为(输入通道数, 高度, 宽度)即CHW
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))      # zip把X和K按第一个维度即通道维度进行配对，得到每个输入通道x和它对应的二维卷积核k

C:\Users\huangleyuan\.conda\envs\geoai\Lib\site-packages\torch\cuda\__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## 验证互相关运算的输出

In [4]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

print(X)
print(K)

corr2d_multi_in(X, K)

tensor([[[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]],

        [[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]])
tensor([[[0., 1.],
         [2., 3.]],

        [[1., 2.],
         [3., 4.]]])


tensor([[ 56.,  72.],
        [104., 120.]])

## 计算多个通道的输出的互相关函数

In [8]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)           # torch.stack把多个张量按第0维即通道维度进行拼接

K = torch.stack((K, K + 1, K + 2), 0)           # 构建3个不同的卷积核(即K, K + 1, K + 2)，按第0维拼接起来
K.shape

torch.Size([3, 3, 3, 2, 2, 2])

In [6]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

## 1x1卷积

In [14]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape             # c_i: 输入通道数；h：高度；w：宽度
    c_o = K.shape[0]             # c_o: 输出通道数（K.shape = (C_out, C_in, K_h, K_w)）
    X = X.reshape((c_i, h * w))          # 将X按通道维度展开成行向量
    K = K.reshape((c_o, c_i))          # 将K按通道维度展开成行向量
    Y = torch.matmul(K, X)          # 矩阵乘法，因为都转置了所以是K乘在前，X乘在后
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)

# abs：求绝对值；assert：如果条件成立（总误差 < 1e-6），程序继续执行；如果条件不成立，抛出 AssertionError，提示两种实现的结果不一致，可能代码有误。
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6       # 这里未抛出错误，可以认为Y1 = Y2